# Multi-format RAG ingestion pipeline
Loads **PDF, DOCX, HTML, CSV and TXT/MD** files from a folder, chunks them, embeds them, and upserts everything into Pinecone.

In [ ]:
%pip install -q pypdf docx2txt beautifulsoup4 lxml rank_bm25 \
    langchain langchain-community langchain-text-splitters langchain-huggingface \
    sentence-transformers pinecone groq python-dotenv pandas tqdm

## Imports

In [ ]:
import os
import csv
from pathlib import Path
from typing import Callable, Dict, List

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    BSHTMLLoader,
)

from pinecone import Pinecone
from groq import Groq

load_dotenv()

## Configuration

In [ ]:
DATA_DIR = "./data"  # folder containing the raw files to ingest (any mix of types below)

EMBEDDING_MODEL_NAME = "thenlper/gte-small"

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = "lab-rag-index"
PINECONE_NAMESPACE = "ns1"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 100

## Multi-format document loaders
Each loader returns a list of `langchain_core.documents.Document`. Add new extensions to `EXTENSION_LOADERS` to support more formats.

In [ ]:
def load_text(path: str) -> List[LangchainDocument]:
    return TextLoader(path, encoding="utf-8").load()


def load_healthcare_csv(path: str) -> List[LangchainDocument]:
    """Loads the healthcare_rag_dataset.csv schema: builds page_content from
    the clinical text fields only (content_text, symptoms, treatments, risk_factors,
    prevention) and keeps the rest of the columns as metadata instead of dumping all
    23 columns into the embedded text (which would bury the signal in noise like
    word_count/reading_level_score and waste context tokens at generation time).
    """
    docs = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            content = "\n".join(
                filter(
                    None,
                    [
                        f"{row.get('title', '')} ({row.get('document_type', '')})",
                        row.get("content_text"),
                        f"Symptoms: {row['symptoms']}" if row.get("symptoms") else None,
                        f"Treatments: {row['treatments']}" if row.get("treatments") else None,
                        f"Risk factors: {row['risk_factors']}" if row.get("risk_factors") else None,
                        f"Prevention: {row['prevention']}" if row.get("prevention") else None,
                    ],
                )
            )
            metadata = {
                "source": path,
                "doc_id": row.get("doc_id"),
                "title": row.get("title"),
                "category": row.get("category"),
                "icd10_code": row.get("icd10_code"),
                "severity_level": row.get("severity_level"),
                "source_name": row.get("source_name"),
                "source_url": row.get("source_url"),
            }
            docs.append(LangchainDocument(page_content=content, metadata=metadata))
    return docs


EXTENSION_LOADERS: Dict[str, Callable[[str], List[LangchainDocument]]] = {
    ".txt": load_text,
    ".md": load_text,
    ".pdf": lambda path: PyPDFLoader(path).load(),
    ".docx": lambda path: Docx2txtLoader(path).load(),
    ".html": lambda path: BSHTMLLoader(path).load(),
    ".htm": lambda path: BSHTMLLoader(path).load(),
    ".csv": load_healthcare_csv,
}


def load_any_file(path: str) -> List[LangchainDocument]:
    ext = Path(path).suffix.lower()
    loader = EXTENSION_LOADERS.get(ext)
    if loader is None:
        print(f"Skipping unsupported file type: {path}")
        return []
    try:
        return loader(path)
    except Exception as exc:
        print(f"Failed to load {path}: {exc}")
        return []


def load_directory(folder: str) -> List[LangchainDocument]:
    all_docs: List[LangchainDocument] = []
    for root, _, files in os.walk(folder):
        for fname in files:
            all_docs.extend(load_any_file(os.path.join(root, fname)))
    return all_docs

## Load every supported file under `DATA_DIR`

In [ ]:
raw_documents = load_directory(DATA_DIR)
print(f"Loaded {len(raw_documents)} source document(s) from {DATA_DIR}")

## Chunking

In [ ]:
MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS,
)

docs_processed = text_splitter.split_documents(raw_documents)
print(f"Split into {len(docs_processed)} chunk(s)")

## Embedding model

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

## Upsert every chunk into Pinecone (batched)

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# One stable id per chunk, reused below for BM25 / fusion / reranking / eval.
chunk_ids = [f"vec{i}" for i in range(len(docs_processed))]
id_to_text = {chunk_id: doc.page_content for chunk_id, doc in zip(chunk_ids, docs_processed)}


def batched(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


for batch in tqdm(list(batched(list(zip(chunk_ids, docs_processed)), UPSERT_BATCH_SIZE))):
    vectors = [
        {
            "id": chunk_id,
            "values": embedding_model.embed_query(doc.page_content),
            "metadata": {
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown"),
            },
        }
        for chunk_id, doc in batch
    ]
    index.upsert(vectors=vectors, namespace=PINECONE_NAMESPACE)

print("Ingestion complete.")

## Quick retrieval + Groq generation smoke test

In [ ]:
groq_client = Groq(api_key=GROQ_API_KEY)

prompt_template = """You are a helpful assistant that answers medical questions using only the provided context.
If the context doesn't contain the answer, say "I don't know".

Context:
{context}

Question: {question}
"""


def ask(question: str, top_k: int = 3) -> str:
    query_vector = embedding_model.embed_query(question)
    results = index.query(
        namespace=PINECONE_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    context = "\n".join(match["metadata"]["text"] for match in results["matches"])
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt_template.format(context=context, question=question)}],
    )
    return response.choices[0].message.content


print(ask("What are the symptoms of diabetes?"))

## Hybrid search: sparse (BM25) + dense retrieval
Dense retrieval (Pinecone + embeddings) is good at semantic similarity but can miss exact keyword/acronym matches. Sparse retrieval (BM25) is the opposite. Combining both, then reranking the merged candidates, is the standard "hybrid search" recipe.

> Runs in the same session as ingestion above, since it reuses `docs_processed`, `chunk_ids` and `id_to_text` from memory rather than re-fetching everything from Pinecone.

### Sparse retriever (BM25)

In [ ]:
import re
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    return re.findall(r"\w+", text.lower())


bm25_corpus_tokens = [tokenize(doc.page_content) for doc in docs_processed]
bm25 = BM25Okapi(bm25_corpus_tokens)


def sparse_search(query: str, top_k: int = 10) -> List[dict]:
    scores = bm25.get_scores(tokenize(query))
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [
        {"id": chunk_ids[i], "score": float(scores[i]), "text": id_to_text[chunk_ids[i]]}
        for i in ranked_idx
    ]

### Dense retriever (Pinecone)

In [ ]:
def dense_search(query: str, top_k: int = 10) -> List[dict]:
    query_vector = embedding_model.embed_query(query)
    results = index.query(
        namespace=PINECONE_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )
    return [
        {"id": match["id"], "score": float(match["score"]), "text": match["metadata"]["text"]}
        for match in results["matches"]
    ]

### Rank fusion (Reciprocal Rank Fusion)
RRF combines multiple ranked lists without needing their scores to be on the same scale (BM25 scores and cosine similarity aren't comparable directly). Each result's fused score is `sum(1 / (k + rank))` across every list it appears in.

In [ ]:
def reciprocal_rank_fusion(rankings: List[List[str]], k: int = 60) -> List[str]:
    fused_scores: Dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused_scores, key=fused_scores.get, reverse=True)


def hybrid_search_fused(query: str, top_k: int = 10, candidate_pool: int = 30) -> List[dict]:
    dense_ids = [r["id"] for r in dense_search(query, candidate_pool)]
    sparse_ids = [r["id"] for r in sparse_search(query, candidate_pool)]
    fused_ids = reciprocal_rank_fusion([dense_ids, sparse_ids])[:top_k]
    return [{"id": doc_id, "text": id_to_text[doc_id]} for doc_id in fused_ids]

### Cross-encoder reranking
RRF fusion is cheap but score-agnostic; a cross-encoder scores each (query, candidate) pair jointly and gives a much better final ordering, at the cost of being slower to run per document — so it's applied only to the small fused candidate set, not the whole corpus.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: List[dict], top_k: int = 5) -> List[dict]:
    pairs = [(query, candidate["text"]) for candidate in candidates]
    scores = reranker.predict(pairs)
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)
    return sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)[:top_k]


def hybrid_search(query: str, top_k: int = 5, candidate_pool: int = 30) -> List[dict]:
    fused = hybrid_search_fused(query, top_k=candidate_pool, candidate_pool=candidate_pool)
    return rerank(query, fused, top_k=top_k)

## Evaluation: NDCG@k across retrieval strategies
NDCG (Normalized Discounted Cumulative Gain) rewards relevant results appearing near the top of the ranking, with graded relevance support (not just relevant/irrelevant). Define `EVAL_SET` below with real chunk ids from `chunk_ids` and their relevance grade per query (e.g. 2 = highly relevant, 1 = somewhat, 0/absent = not relevant), then compare dense-only, sparse-only, RRF-fused and reranked hybrid search.

In [ ]:
import math


def ndcg_at_k(relevances: List[int], k: int) -> float:
    relevances = relevances[:k]
    dcg = sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances))
    ideal = sorted(relevances, reverse=True)
    idcg = sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
# Real test cases: each maps a natural question to the doc_id in
# healthcare_rag_dataset.csv that should answer it. Extend this list with more
# doc_id/query pairs as you ingest more data — no hardcoded chunk indices.
TEST_QUERIES = [
    ("What are the symptoms of conjunctivitis (pink eye)?", "DOC-E6C0C42D"),
    ("What are common symptoms of allergic rhinitis?", "DOC-D8F798B2"),
    ("What does eczema look like and what are its symptoms?", "DOC-E911986D"),
    ("What are the symptoms of hyperlipidemia?", "DOC-1427E31F"),
    ("What are the symptoms of asthma?", "DOC-2117F5AF"),
    ("What are the symptoms of iron deficiency anemia?", "DOC-6FF549B5"),
    ("What are the symptoms of type 2 diabetes?", "DOC-1730475D"),
    ("What are the symptoms of a urinary tract infection?", "DOC-64EEB1E0"),
]


def chunk_ids_for_doc(doc_id: str) -> List[str]:
    """All chunk ids whose source metadata came from this doc_id (a document
    can span several chunks after splitting)."""
    return [
        chunk_id
        for chunk_id, doc in zip(chunk_ids, docs_processed)
        if doc.metadata.get("doc_id") == doc_id
    ]


EVAL_SET = [
    {"query": query, "relevant_grades": {cid: 2 for cid in chunk_ids_for_doc(doc_id)}}
    for query, doc_id in TEST_QUERIES
]
EVAL_SET = [example for example in EVAL_SET if example["relevant_grades"]]

missing = len(TEST_QUERIES) - len(EVAL_SET)
if missing:
    print(f"Warning: {missing} test doc_id(s) not found in docs_processed — did ingestion run?")

K = 5

RETRIEVAL_METHODS = {
    "dense_only": lambda q: [r["id"] for r in dense_search(q, K)],
    "sparse_only": lambda q: [r["id"] for r in sparse_search(q, K)],
    "hybrid_fused": lambda q: [r["id"] for r in hybrid_search_fused(q, top_k=K)],
    "hybrid_reranked": lambda q: [r["id"] for r in hybrid_search(q, top_k=K)],
}


def evaluate_method(method_fn, eval_set, k) -> float:
    scores = []
    for example in eval_set:
        ranked_ids = method_fn(example["query"])
        relevances = [example["relevant_grades"].get(doc_id, 0) for doc_id in ranked_ids]
        scores.append(ndcg_at_k(relevances, k))
    return sum(scores) / len(scores) if scores else 0.0


results_df = pd.DataFrame(
    [
        {"method": name, f"NDCG@{K}": evaluate_method(fn, EVAL_SET, K)}
        for name, fn in RETRIEVAL_METHODS.items()
    ]
)
results_df

### NDCG regression test
Runs after the whole notebook (ingestion + all retrieval strategies) and fails loudly if hybrid search's NDCG@5 drops below a floor — a real pass/fail check, not just a printed table. Re-run this cell any time after re-ingesting data or tweaking `RECIPROCAL_RANK_FUSION`/reranker settings to catch regressions.

In [ ]:
NDCG_FLOOR = 0.7  # hybrid_reranked should reliably surface the right doc for these clear-cut queries

assert EVAL_SET, "EVAL_SET is empty — run the ingestion cells first so TEST_QUERIES' doc_ids exist."

failures = []
for _, row in results_df.iterrows():
    score = row[f"NDCG@{K}"]
    status = "PASS" if score >= NDCG_FLOOR else "FAIL"
    if row["method"] == "hybrid_reranked" and status == "FAIL":
        failures.append(f"{row['method']}: NDCG@{K}={score:.3f} < floor {NDCG_FLOOR}")
    print(f"[{status}] {row['method']:<16} NDCG@{K} = {score:.3f}")

assert not failures, "NDCG regression detected:\n" + "\n".join(failures)
print("\nAll NDCG checks passed.")